In [ ]:
import os
import pandas as pd
from datasets import Dataset

from utils.bq import get_client, query_to_df
from utils.hub_card import push_dataset_card

client, PROJECT_NAME = get_client()

In [ ]:
# Lab results from hosp.labevents for cohort patients.
# labevents has no ed_stay_id — joined via subject_id + time window.
# Window covers:
#   - ED-only patients: ed_intime to ed_outtime
#   - Admitted patients: ed_intime to first_icu_intime (if ICU transfer occurred) or dischtime
# Labs ordered after ICU transfer are excluded from the window entirely.
# order_time = charttime (when the order was placed / specimen collected)
# result_time = storetime (when the result was filed)
# ordered_location: 'ed' if order placed during ED window, 'ward' after ED discharge
# result_after_icu_transfer: True if result_time >= first_icu_intime — lab was ordered before
#   ICU transfer but result came back after (retrospective label, not a state feature)
# hadm_id is NULL for ED-only patients, populated for admitted patients.
# NOTE: This query may take several minutes due to labevents table size.

query_labs = f"""
WITH cohort_subjects AS (
  SELECT
    ed_stay_id,
    subject_id,
    hadm_id,
    ed_intime,
    ed_outtime,
    first_icu_intime,
    -- window_end caps at ICU transfer for admitted patients who went to ICU
    CASE
      WHEN hadm_id IS NOT NULL AND first_icu_intime IS NOT NULL THEN first_icu_intime
      WHEN hadm_id IS NOT NULL THEN dischtime
      ELSE ed_outtime
    END AS window_end
  FROM `{PROJECT_NAME}.rl_project.cohort_base`
)
SELECT
  cs.ed_stay_id,
  le.subject_id,
  cs.hadm_id,
  le.labevent_id,
  le.itemid,
  dl.label,
  dl.fluid,
  dl.category,
  le.charttime AS order_time,
  le.storetime AS result_time,
  le.flag,
  CASE
    WHEN le.charttime <= cs.ed_outtime THEN 'ed'
    ELSE 'ward'
  END AS ordered_location,
  CASE
    WHEN cs.first_icu_intime IS NOT NULL AND le.storetime >= cs.first_icu_intime THEN TRUE
    ELSE FALSE
  END AS result_after_icu_transfer
FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
INNER JOIN cohort_subjects cs
  ON le.subject_id = cs.subject_id
  AND le.charttime >= cs.ed_intime
  AND le.charttime < cs.window_end
INNER JOIN `physionet-data.mimiciv_3_1_hosp.d_labitems` dl
  ON le.itemid = dl.itemid
ORDER BY cs.ed_stay_id, le.charttime
"""

print("Running labs query (may take several minutes)...")
df_labs = query_to_df(client, query_labs)
print(f"Shape: {df_labs.shape}")
print(f"Unique ED visits with labs: {df_labs['ed_stay_id'].nunique():,}")
print(f"\nOrdered location breakdown:\n{df_labs['ordered_location'].value_counts()}")
print(f"\nResult after ICU transfer: {df_labs['result_after_icu_transfer'].sum():,} rows")
print(f"\nCategory x fluid combinations: {df_labs.groupby(['category', 'fluid']).ngroups}")
df_labs.head()

In [ ]:
DESCRIPTION = (
    "Laboratory results from hosp.labevents for cohort patients during their stay window. "
    "Grouped by category x fluid (19 unique combinations) rather than individual test label "
    "to reduce action space. Each result includes the abnormal flag for worst-case aggregation. "
    "Intended as the primary lab state feature source."
)

ds = Dataset.from_pandas(df_labs, split='labs_base')
del df_labs  # Do this for memory issues

ds.push_to_hub("ADS599-Capstone/raw_data", config_name="labs_base", data_dir="lab_events")
push_dataset_card("ADS599-Capstone/raw_data", config_name="labs_base", split="labs_base", data_dir="lab_events", description=DESCRIPTION)
print("Pushed labs_base to HuggingFace Hub.")